In [1]:
import numpy as np
import pandas as pd

from prokbert.data_collator import SegmentDataCollator
from prokbert.sequtils import load_contigs, segment_sequences
from prokbert.sequence_dataset import SequenceDataset, EmbeddingDataset
from prokbert.dataset import RandomSegmentDataset, ContiguousSegmentDataset

2026-10-26 13:45:26,163 - INFO - TensorFlow version 2.21.0 available.


In [2]:
base_dir = "/home/datasets/experiments/experiment-bioinfo/fine-tuning"

# Dataset

## New implementation

In [3]:
train_seq_ds = SequenceDataset()

train_save_dir = f"{base_dir}/example-train-dataset/"
train_seq_dataset_save_dir = f"{base_dir}/example-train-sequence-dataset"
train_seq_ds.create_dataset([train_save_dir], save_dir=train_seq_dataset_save_dir)

print("Total length:", len(train_seq_ds.sequence))

2026-10-26 13:45:30,012 - INFO - Dataset saved to '/home/datasets/experiments/experiment-bioinfo/fine-tuning/example-train-sequence-dataset'.


Total length: 116680168


In [4]:
pd.DataFrame(train_seq_ds.metadata).head()

,genome_id,contig_id,sequence_id,coordinate,orientation,description,label
0,None,PHAA0000323,0,"(0, 53124)",forward,None,0
1,None,PHAA0000323,1,"(53124, 106248)",reverse,None,0
2,None,PHAA0000761,2,"(106248, 177813)",forward,None,1
3,None,PHAA0000761,3,"(177813, 249378)",reverse,None,1
4,None,PHAA0000840,4,"(249378, 302774)",forward,None,0


In [5]:
test_seq_ds = SequenceDataset()

test_seq_dataset_save_dir = f"{base_dir}/example-test-sequence-dataset"
test_save_dir = f"{base_dir}/example-test-dataset/"

test_seq_ds.create_dataset([test_save_dir], save_dir=test_seq_dataset_save_dir)

print("Total length:", len(test_seq_ds.sequence))

2026-10-26 13:45:30,248 - INFO - Dataset saved to '/home/datasets/experiments/experiment-bioinfo/fine-tuning/example-test-sequence-dataset'.


Total length: 24854660


In [6]:
pd.DataFrame(test_seq_ds.metadata).head()

,genome_id,contig_id,sequence_id,coordinate,orientation,description,label
0,None,PHAA0000415,0,"(0, 168903)",forward,None,1
1,None,PHAA0000415,1,"(168903, 337806)",reverse,None,1
2,None,PHAA0000088,2,"(337806, 398748)",forward,None,0
3,None,PHAA0000088,3,"(398748, 459690)",reverse,None,0
4,None,PHAA0000253,4,"(459690, 521360)",forward,None,0


## Old implementation

In [7]:
train_save_path = f"{base_dir}/example-train-dataset/phastyle-sequencedb-bachlip-training.fasta"
test_save_path = f"{base_dir}/example-test-dataset/phastyle-sequencedb-bachlip-validation.fasta"

In [8]:
train_dataset = load_contigs(
    [train_save_path],
    IsAddHeader=True,
    adding_reverse_complement=False,
    AsDataFrame=True,
    to_uppercase=True,
    is_add_sequence_id=True,
)

2026-10-26 13:45:30,403 - INFO - Loading sequence data into memory!


In [9]:
train_dataset.head()

,sequence_id,fasta_id,description,source_file,sequence,orientation
0,0,PHAA0000323,PHAA0000323 empy description,/home/datasets/experiments/experiment-bioinfo/...,TGCGGCCGCCCCATCCTGTACGGGTTTCCAAGTCGATCGGAGGGCA...,forward
1,1,PHAA0000323,PHAA0000323 empy description,/home/datasets/experiments/experiment-bioinfo/...,TAACCGACCGGTCAGGAAGGTACCCCTGGGGGGGTACTCCCCCTGG...,forward
2,2,PHAA0000761,PHAA0000761 empy description,/home/datasets/experiments/experiment-bioinfo/...,ATGGCTTCGACTCAGACTCCAGCCGTCGGCAAGACCACGGCCATCG...,forward
3,3,PHAA0000761,PHAA0000761 empy description,/home/datasets/experiments/experiment-bioinfo/...,GGGCGCAGGATACCAGCAGGGTTGACTTACCCGACTCACTCGGGTA...,forward
4,4,PHAA0000840,PHAA0000840 empy description,/home/datasets/experiments/experiment-bioinfo/...,TGCGGCTGAGCCATCGTGTACGGGTTTCCAAGTCCATCAGAGCCGG...,forward


In [10]:
test_dataset = load_contigs(
    [test_save_path],
    IsAddHeader=True,
    adding_reverse_complement=False,
    AsDataFrame=True,
    to_uppercase=True,
    is_add_sequence_id=True,
)

2026-10-26 13:45:31,333 - INFO - Loading sequence data into memory!


In [11]:
test_dataset.head()

,sequence_id,fasta_id,description,source_file,sequence,orientation
0,0,PHAA0000415,PHAA0000415 empy description,/home/datasets/experiments/experiment-bioinfo/...,AATTTTCCTTATTAGGCCGCAAGGGCCTTCATAGTTTTAGCGATTT...,forward
1,1,PHAA0000415,PHAA0000415 empy description,/home/datasets/experiments/experiment-bioinfo/...,ATGTACAATATTAAATGCCTGACCAAAAACGAACAAGCTGAAATTG...,forward
2,2,PHAA0000088,PHAA0000088 empy description,/home/datasets/experiments/experiment-bioinfo/...,GTTTCAACATGTCTAAGGATATGGTTTATCATTTTGAAAAATATAA...,forward
3,3,PHAA0000088,PHAA0000088 empy description,/home/datasets/experiments/experiment-bioinfo/...,CATGAAGCATTAAAAACACAGAATTACCGAAGATCGTCCAAGAGCA...,forward
4,4,PHAA0000253,PHAA0000253 empy description,/home/datasets/experiments/experiment-bioinfo/...,GTTTCAACATGTCTAAGGATATGGTTTATCATTTTGAAAAATATAA...,forward


# Segment creation

### Contiguous segmentation

In [12]:
min_segment_length = 1000
max_segment_length = 2000
segmentation_type = "contiguous"

In [13]:
test_segment_dataset_new = ContiguousSegmentDataset(
    sequence_dataset=test_seq_ds,
    min_segment_length=min_segment_length,
    max_segment_length=max_segment_length,
    return_sequence=True
)

In [14]:
for i, segment in enumerate(iter(test_segment_dataset_new)):
    if i >= 3: break
    print(segment)

{'segment_id': 0, 'contig_id': 'PHAA0000415', 'genome_id': None, 'sequence_id': 0, 'absolute_coordinate': (0, 2000), 'relative_coordinate': (0, 2000), 'orientation': 'forward', 'label': 1, 'sequence': 'AATTTTCCTTATTAGGCCGCAAGGGCCTTCATAGTTTTAGCGATTTGGGAAACTTCATCATCACTTAAAGAGTTGCGATAACCGATGAAGTCGGAAACAATACGGAATTTCTTGGTAAACTCAGCAACCATTTTATCACTGTTTTTTGAAGCATTATTTGATAATACATCAAAAAGATTAGTTACTGTCCAAATGTCATGACCGATGGTATCTTTTCCACCATTAAAATATACACCCTGTAATGAACTAACCATATTAGCGAGTCGTGTATATTCTTCAGAAACTTCATCTATACTGAAGTACTTCATCATAAAATCTAACTCAGGATACTTGATAATTTTATCAATATATCGTTTAGCTGAACTTGAATAACCTACATACTTATCATAATCTACATCATCAAAAGCATCTACATATAAATCACGCAAAGCTTCAAAAATACATTGGCACTGACCGAGTTCTTTTACCTTTTTCTGTAAAAGCGGACGAATAACATAAAATTCATTAATGCCAATAAGATTAGCCATACGAATCAAAATATTCATAGATGGATGACAAAGAGATGTAGTACCATCCATAGAGAAAATATCAGAACGATGCATATACGCTACATAACCAGTAATTTCATCTGCTTCTGATGTGAGGCGTAAATAATTCCTCTTTTCCCAGCGCCCGTCTTTAATTTCAAACTTAAACGCTGTAGCAGCTTTAGGACGAGGAGCTTTACTTTTAACTACCTTTGGAATATAACTTTTTACTAAAGCTTCAATTTCTGACAAATAATGAATGTTAACTTCA

In [15]:
segmentation_params = {
    "max_length": max_segment_length,
    "min_length": min_segment_length,
    "type": segmentation_type,
}

test_segment_dataset_old = segment_sequences(test_dataset, segmentation_params, AsDataFrame=True)

2026-10-26 13:45:31,938 - INFO - Checking input DataFrame!
2026-10-26 13:45:31,940 - INFO - Checking input sequence_id is valid primary key in the DataFrame


In [16]:
test_segment_dataset_old.head()

,segment_id,sequence_id,segment_start,segment_end,segment
0,0,0,0,2000,AATTTTCCTTATTAGGCCGCAAGGGCCTTCATAGTTTTAGCGATTT...
1,1,0,2000,4000,TTCAGGATTTCCATTAAGTGCATGAGCATCAATCATGTTAGTAATC...
2,2,0,4000,6000,TTGAGAACTTTACCACGCAATGGATAACCACCGTGAAGTTCTTTAT...
3,3,0,6000,8000,CACTTCATCAAAGAACCCAGCTTCGTTAGTAGAGAGCCAGATAAAG...
4,4,0,8000,10000,AAATTAGGTCTGTCATAACCTACATGTAATAACCACTCTTCACTCC...


Compare the old and new segment datasets

In [17]:
rows = []
for seg in test_segment_dataset_new:
    start, end = seg["relative_coordinate"]
    rows.append({
        "sequence_id": seg["sequence_id"],
        "segment_start": start,
        "segment_end": end,
        "segment": seg.get("sequence"),     # None if the dataset was created without return_sequence=True
    })
new_df = pd.DataFrame(rows)
new_df.head()

,sequence_id,segment_start,segment_end,segment
0,0,0,2000,AATTTTCCTTATTAGGCCGCAAGGGCCTTCATAGTTTTAGCGATTT...
1,0,2000,4000,TTCAGGATTTCCATTAAGTGCATGAGCATCAATCATGTTAGTAATC...
2,0,4000,6000,TTGAGAACTTTACCACGCAATGGATAACCACCGTGAAGTTCTTTAT...
3,0,6000,8000,CACTTCATCAAAGAACCCAGCTTCGTTAGTAGAGAGCCAGATAAAG...
4,0,8000,10000,AAATTAGGTCTGTCATAACCTACATGTAATAACCACTCTTCACTCC...


In [18]:
old_df = test_segment_dataset_old[["sequence_id", "segment_start", "segment_end", "segment"]]
key = ["sequence_id", "segment_start", "segment_end"]
old_df.head()

,sequence_id,segment_start,segment_end,segment
0,0,0,2000,AATTTTCCTTATTAGGCCGCAAGGGCCTTCATAGTTTTAGCGATTT...
1,0,2000,4000,TTCAGGATTTCCATTAAGTGCATGAGCATCAATCATGTTAGTAATC...
2,0,4000,6000,TTGAGAACTTTACCACGCAATGGATAACCACCGTGAAGTTCTTTAT...
3,0,6000,8000,CACTTCATCAAAGAACCCAGCTTCGTTAGTAGAGAGCCAGATAAAG...
4,0,8000,10000,AAATTAGGTCTGTCATAACCTACATGTAATAACCACTCTTCACTCC...


In [19]:
# 2. same number of segments, no duplicates
print(f"old: {len(old_df)} segments, new: {len(new_df)} segments")
print(f"duplicates: old {old_df.duplicated(key).sum()}, new {new_df.duplicated(key).sum()}")


old: 12420 segments, new: 12420 segments
duplicates: old 0, new 0


In [20]:
# 3. same segments: an outer merge shows which ones exist on only one side
merged = old_df.merge(new_df, on=key, how="outer", suffixes=("_old", "_new"), indicator=True)
print(merged["_merge"].value_counts())      # ideally only "both"

_merge
both          12420
left_only         0
right_only        0
Name: count, dtype: int64


In [21]:
# 4. same sequences for the segments both have
both = merged[merged["_merge"] == "both"]
print(f"sequence mismatches: {(both['segment_old'] != both['segment_new']).sum()}")

sequence mismatches: 0


### Random segmentation

In [32]:
min_segment_length = 1000
max_segment_length = 2000
segmentation_type = "random"
coverage = 2
seed = 42

In [33]:
train_segment_dataset_new = RandomSegmentDataset(
    sequence_dataset=train_seq_ds,
    min_segment_length=min_segment_length,
    max_segment_length=max_segment_length,
    return_sequence=True,
    coverage=coverage,
    seed=seed,
)

In [34]:
segmentation_params = {
    "max_length": max_segment_length,
    "min_length": min_segment_length,
    "type": segmentation_type,
    "coverage": coverage
}

np.random.seed(seed)
train_segment_dataset_old = segment_sequences(train_dataset, segmentation_params, AsDataFrame=True)

2026-10-26 13:47:30,271 - INFO - Checking input DataFrame!
2026-10-26 13:47:30,272 - INFO - Checking input sequence_id is valid primary key in the DataFrame
2026-10-26 13:47:30,387 - INFO - Sampling 116681 segments from 1798 sequences.
2026-10-26 13:47:30,424 - INFO - Too short segment, skip! Sampled segment: AGTCATCGAGGAAGAAATGGAGAGCGCCGCCTGTATGGGCGGCGTATTCCCGGTGTCGCGGCATATTCCAGGCCGCGAGGTTCTCGGGAACGAAGTCGGTGGCTGTCAAGTCGGGAATCTCATGAGCGGATGTGCTCGGGAACGTCATCCGCAGGTTGAGAACGTCGAACTTGCCCGGTTGCGAGGCCCAGAAGGCACTCGAGCGAGTGCCGTACACGGTTGCCTCACCCCTATCTATGAGCCCGCGCACTGGCGCGGGCGAAGGACTCGCGGTTGCGAGTCCAATCTCAGACCGGCCCGGTGGCCGGTCTTCTAGAAGAGCGCCTGGTGGCGCTCTAGATAGGGCCCTCTTGGGCCCTCATAATATAAGTACCTGAGTATTCCTTGACATCCACCTGGGTGTGTGGTAGCTCACGGATGACTCCCAGTTTCGCGAGCGTTAATACAGGCAACCCTGACGGGTTGTACCTTGCCGCGCAGCCCGTCAGGGGCTGCAGACGGCGACCGAAGGAAAGGCTCATGAAGTACCTGCTGCGAGATCTGATCAACGTCCCATTCATCCTGCTCGACCGCGCTCGCGGGCGGGCGATCGACTGGATCGAGGGTGACGACGACCACAGCATGAGGCCAGGCTCTGAGAGGTGCTGAGCCGATTCCGGGGGTACAACTACCCGCGTGAGCGGGGATCT

In [35]:
train_segment_dataset_old.head()

,segment_id,sequence_id,segment_start,segment_end,segment
0,0,0,646,2646,CCGAGCACATCCGCTCATGAGATTCCCGACTTGACAGCCACCGACT...
1,1,0,983,2983,GCACCCCCGACACCTACGACTTCTGTTTCGACGGCCTGCCGACTGG...
2,2,0,1357,3357,GGTCCGCTGGTGGCGGCGGTGGCGGGGCCCCCAAGGGCACCGGTAG...
3,3,0,1952,3952,GGTCGGCGGCAAGGCCGCGATGGACTACATGCCGGTGCAGATCACC...
4,4,0,3212,5212,GACATCCGGGAAGGTCGTTGACATGACCGACATGCTCAAGCAGCGC...


In [36]:
train_segment_dataset_old.shape

(114849, 5)

In [39]:
rows = []
for seg in train_segment_dataset_new:
    start, end = seg["relative_coordinate"]
    rows.append({
        "segment_id": seg["segment_id"],
        "sequence_id": seg["sequence_id"],
        "segment_start": start,
        "segment_end": end,
        "segment": seg["sequence"],
    })
new_df = pd.DataFrame(rows)
new_df.head()

,segment_id,sequence_id,segment_start,segment_end,segment
0,0,0,646,2646,CCGAGCACATCCGCTCATGAGATTCCCGACTTGACAGCCACCGACT...
1,1,0,983,2983,GCACCCCCGACACCTACGACTTCTGTTTCGACGGCCTGCCGACTGG...
2,2,0,1357,3357,GGTCCGCTGGTGGCGGCGGTGGCGGGGCCCCCAAGGGCACCGGTAG...
3,3,0,1952,3952,GGTCGGCGGCAAGGCCGCGATGGACTACATGCCGGTGCAGATCACC...
4,4,0,3212,5212,GACATCCGGGAAGGTCGTTGACATGACCGACATGCTCAAGCAGCGC...


In [40]:
old_df = train_segment_dataset_old[["segment_id", "sequence_id", "segment_start", "segment_end", "segment"]]
key = ["segment_id", "sequence_id", "segment_start", "segment_end"]
old_df.head()

,segment_id,sequence_id,segment_start,segment_end,segment
0,0,0,646,2646,CCGAGCACATCCGCTCATGAGATTCCCGACTTGACAGCCACCGACT...
1,1,0,983,2983,GCACCCCCGACACCTACGACTTCTGTTTCGACGGCCTGCCGACTGG...
2,2,0,1357,3357,GGTCCGCTGGTGGCGGCGGTGGCGGGGCCCCCAAGGGCACCGGTAG...
3,3,0,1952,3952,GGTCGGCGGCAAGGCCGCGATGGACTACATGCCGGTGCAGATCACC...
4,4,0,3212,5212,GACATCCGGGAAGGTCGTTGACATGACCGACATGCTCAAGCAGCGC...


In [41]:
# 1. same number of segments, no duplicates
print(f"old: {len(old_df)} segments, new: {len(new_df)} segments")
print(f"duplicates: old {old_df.duplicated(key).sum()}, new {new_df.duplicated(key).sum()}")

old: 114849 segments, new: 114849 segments
duplicates: old 0, new 0


In [42]:
# 2. same segments with the same ids: an outer merge shows which ones exist on only one side
merged = old_df.merge(new_df, on=key, how="outer", suffixes=("_old", "_new"), indicator=True)
print(merged["_merge"].value_counts())      # ideally only "both"

_merge
both          114849
left_only          0
right_only         0
Name: count, dtype: int64


In [43]:
# 3. same sequences for the segments both have
both = merged[merged["_merge"] == "both"]
print(f"sequence mismatches: {(both['segment_old'] != both['segment_new']).sum()}")

sequence mismatches: 0
